# 04 — Model Evaluation

Evaluates the trained ONNX model on the held-out test set and selects confidence thresholds.

**Target: AUC ≥ 0.90 on the test set.**

The thresholds used in `ml/serve/app.py`:
- `≥ 0.85` → AutoApprove (element skips the review queue)
- `0.45–0.84` → NeedsReview (staff sees yellow badge)
- `< 0.45` → AutoReject badge (staff sees red badge, still has final say)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import onnxruntime as ort
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

test_df = pd.read_csv('/kaggle/working/test.csv')

session    = ort.InferenceSession('/kaggle/working/nature_classifier.onnx')
input_name = session.get_inputs()[0].name
print(f'ONNX input tensor name: "{input_name}"')

In [ ]:
# --- Run inference on test set ---
def predict(paths, batch_size=64):
    preds = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        imgs = []
        for p in batch:
            try:
                img = tf.image.resize(
                    tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
                    [224, 224]
                ).numpy()
                imgs.append(preprocess_input(img))
            except Exception:
                imgs.append(np.zeros((224, 224, 3), dtype='float32'))
        arr = np.stack(imgs).astype('float32')
        preds.extend(session.run(None, {input_name: arr})[0].flatten().tolist())
    return np.array(preds)

y_pred = predict(test_df['path'].tolist())
y_true = test_df['label'].values
print(f'Predictions: min={y_pred.min():.3f}  max={y_pred.max():.3f}  mean={y_pred.mean():.3f}')

In [ ]:
# --- ROC Curve + AUC ---
auc = roc_auc_score(y_true, y_pred)
print(f'AUC: {auc:.4f}  (target ≥ 0.90)')

fpr, tpr, _ = roc_curve(y_true, y_pred)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Classification report at each threshold ---
for threshold in [0.85, 0.45]:
    print(f'\n=== Threshold = {threshold} ===')
    print(classification_report(y_true, (y_pred >= threshold).astype(int),
                                 target_names=['Non-nature', 'Nature element']))

In [ ]:
# --- Confusion matrix at threshold=0.85 ---
cm = confusion_matrix(y_true, (y_pred >= 0.85).astype(int))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-nature', 'Nature'],
            yticklabels=['Non-nature', 'Nature'])
plt.title('Confusion matrix (threshold=0.85)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

In [ ]:
# --- Worst false positives (non-plant predicted as plant ≥ 0.85) ---
fp_idx = np.where((y_true == 0) & (y_pred >= 0.85))[0]
fp_idx = fp_idx[np.argsort(y_pred[fp_idx])[::-1]][:10]
print(f'False positives at 0.85: {len(fp_idx)}')

if len(fp_idx):
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for ax, i in zip(axes.flat, fp_idx):
        try:
            from PIL import Image
            img = Image.open(test_df.iloc[i]['path']).resize((112, 112))
            ax.imshow(img)
            ax.set_title(f'{y_pred[i]:.2f}', fontsize=8)
        except Exception:
            pass
        ax.axis('off')
    plt.suptitle('False positives (non-nature → predicted nature)')
    plt.tight_layout()
    plt.show()